In [0]:
%pip install PyPDF2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import os

path = "/Workspace/FileStore/hansel_and_gretel_story.pdf"

print(os.path.exists(path))

True


In [0]:
import PyPDF2

text=""

with open(path, "rb") as file:
    reader=PyPDF2.PdfReader(file)
    for page in reader.pages:
        text+=page.extract_text()


    

In [0]:
print(text)

Hansel and Gretel
 
Once upon a time, in a small village near a dark forest, lived a poor woodcutter with his two children,
Hansel and Gretel. Their stepmother, worried about having too little food, convinced the woodcutter to
leave the children in the forest. Hansel, overhearing the plan, quietly gathered small white pebbles and
dropped them along the path as they were led deep into the woods. When night fell, the children
followed the shining pebbles back home safely.
But the next day, the stepmother insisted again. This time Hansel could not gather pebbles, so he used
breadcrumbs instead. However, birds ate the crumbs, and the children were lost in the forest. Hungry
and afraid, they wandered until they found a house made of gingerbread, cakes, and candy. Delighted,
they began to eat pieces of it.
An old woman came out and invited them inside. She seemed kind, but she was actually a wicked witch
who lured children to eat them. She locked Hansel in a cage and forced Gretel to cook, p

In [0]:
print(type(text))

<class 'str'>


In [0]:
chunk_size=500
overlap=50

chunks=[]
for doc in text:
    start=0
    while start<len(text):
        chunk=text[start:start+chunk_size]
        chunks.append(chunk)
        start=start+chunk_size-overlap
        

In [0]:
print(len(chunks))

5900


In [0]:
%pip install sentence-transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/local_disk0/.ephemeral_nfs/envs/pythonEnv-7fde85e7-25f1-4d4e-a974-67356d749c3f/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
embeddings=model.encode(chunks)

In [0]:
print(len(embeddings))
print(len(embeddings[0]))

5900
384


In [0]:
%pip install faiss-cpu

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import faiss
import numpy as np

In [0]:
embedding_matrix = np.array(embeddings).astype("float32")

In [0]:
dim = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embedding_matrix)

In [0]:
print(index.ntotal)

5900


In [0]:
query = "who is witch"

query_embedding = model.encode([query]).astype("float32")

k = 3

distances, indices = index.search(query_embedding, k)

In [0]:
print(distances,distances)

[[1.1386757 1.1386757 1.1386757]] [[1.1386757 1.1386757 1.1386757]]


In [0]:
retrieved_chunks = [chunks[i] for i in indices[0]]

In [0]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question based only on the context below:

Context:
{context}

Question:
{query}
"""

In [0]:
%pip install groq

  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7fde85e7-25f1-4d4e-a974-67356d749c3f
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from groq import Groq

client = Groq(api_key="Grok_api_key")

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [0]:
answer = response.choices[0].message.content
print(answer)

The witch is a character from a story involving Hansel and Gretel. She is a woman who lured children to eat them, trapped Hansel in a cage, and forced Gretel to cook, planning to fatten him up.
